In [ ]:
!pip install -q pandas scikit-learn flask joblib

In [ ]:
import pandas as pd

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00267/data_banknote_authentication.txt"

columns = [
    "variance",
    "skewness",
    "curtosis",
    "entropy",
    "class"
]

data = pd.read_csv(url, header=None, names=columns)

print("Dataset loaded successfully!")
print("Dataset shape:", data.shape)
display(data.head())
print("\nClass distribution:")
print(data["class"].value_counts())

Dataset loaded successfully!
Dataset shape: (1372, 5)


,variance,skewness,curtosis,entropy,class
0,3.62160,8.6661,-2.8073,-0.44699,0
1,4.54590,8.1674,-2.4586,-1.46210,0
2,3.86600,-2.6383,1.9242,0.10645,0
3,3.45660,9.5228,-4.0112,-3.59440,0
4,0.32924,-4.4552,4.5718,-0.98880,0



Class distribution:
class
0    762
1    610
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report
import joblib

# Separate features and target
X = data.drop("class", axis=1)
y = data["class"]

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Create Linear SVM model
model = make_pipeline(
    StandardScaler(),
    SVC(kernel="linear")
)

# Train model
model.fit(X_train, y_train)

# Predict test data
y_pred = model.predict(X_test)

# Evaluate model
print("Model Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Save model
joblib.dump(model, "/content/banknote_model.pkl")

print("Model saved successfully!")

Model Accuracy: 0.9745454545454545

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.95      0.98       153
           1       0.95      1.00      0.97       122

    accuracy                           0.97       275
   macro avg       0.97      0.98      0.97       275
weighted avg       0.98      0.97      0.97       275

Model saved successfully!


In [ ]:
from pathlib import Path

Path("/content/templates").mkdir(exist_ok=True)

app_code = '''
from flask import Flask, render_template, request
import joblib

app = Flask(__name__)
model = joblib.load("/content/banknote_model.pkl")

@app.route("/", methods=["GET", "POST"])
def home():
    result = None
    error = None

    if request.method == "POST":
        try:
            features = [[
                float(request.form["variance"]),
                float(request.form["skewness"]),
                float(request.form["curtosis"]),
                float(request.form["entropy"])
            ]]

            prediction = int(model.predict(features)[0])

            if prediction == 0:
                result = "Genuine Banknote"
            else:
                result = "Counterfeit Banknote"

        except (ValueError, TypeError):
            error = "Please enter valid numerical values."

    return render_template(
        "index.html",
        result=result,
        error=error
    )

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=False)
'''

Path("/content/app.py").write_text(app_code)

print("Flask application file created successfully!")


Flask application file created successfully!


In [ ]:
html_code = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <title>Banknote Detection</title>

    <style>
        body {
            font-family: Arial, sans-serif;
            background: #eaf0f6;
            padding: 30px 15px;
        }

        .container {
            max-width: 430px;
            margin: auto;
            background: white;
            padding: 25px;
            border-radius: 12px;
            box-shadow: 0 4px 15px #0002;
        }

        h1 {
            color: #183b56;
            text-align: center;
        }

        label {
            display: block;
            margin-top: 15px;
            font-weight: bold;
        }

        input {
            width: 100%;
            box-sizing: border-box;
            padding: 10px;
            margin-top: 6px;
            border: 1px solid #aaa;
            border-radius: 5px;
        }

        button {
            width: 100%;
            padding: 12px;
            margin-top: 22px;
            background: #1769aa;
            color: white;
            border: none;
            border-radius: 5px;
            font-size: 16px;
            cursor: pointer;
        }

        .result {
            padding: 12px;
            margin-top: 20px;
            text-align: center;
            background: #e7f4e8;
            border-radius: 5px;
            font-weight: bold;
        }

        .error {
            color: red;
        }
    </style>
</head>

<body>
    <div class="container">
        <h1>Banknote Detection</h1>
        <p>Enter the four numerical features.</p>

        <form method="POST">
            <label>Variance</label>
            <input type="number" step="any"
                   name="variance" required>

            <label>Skewness</label>
            <input type="number" step="any"
                   name="skewness" required>

            <label>Curtosis</label>
            <input type="number" step="any"
                   name="curtosis" required>

            <label>Entropy</label>
            <input type="number" step="any"
                   name="entropy" required>

            <button type="submit">Predict Banknote</button>
        </form>

        {% if result %}
        <div class="result">Prediction: {{ result }}</div>
        {% endif %}

        {% if error %}
        <p class="error">{{ error }}</p>
        {% endif %}
    </div>
</body>
</html>
"""

Path("/content/templates/index.html").write_text(html_code)

print("HTML page created successfully!")

HTML page created successfully!


In [ ]:
!pip install -q pyngrok

In [ ]:
import getpass
import threading
import time
from pyngrok import ngrok

# Enter your ngrok token privately
token = getpass.getpass("Enter your ngrok authtoken: ")
ngrok.set_auth_token(token)

# Start the Flask application
def run_flask():
    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )

thread = threading.Thread(target=run_flask, daemon=True)
thread.start()

# Wait for Flask to start
time.sleep(2)

# Create a public tunnel
public_url = ngrok.connect(5000).public_url

print("Your Banknote Detection Website:")
print(public_url)

Enter your ngrok authtoken: ··········


Exception in thread Thread-3 (run_flask):
Traceback (most recent call last):
  File "/usr/lib/python3.13/threading.py", line 1044, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/threading.py", line 995, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1131/17127049.py", line 12, in run_flask
    app.run(
    ^^^
NameError: name 'app' is not defined


Your Banknote Detection Website:
https://canned-antler-outburst.ngrok-free.dev


In [ ]:
# Load the Flask application from app.py
from app import app

print("Flask application loaded successfully!")

Flask application loaded successfully!


In [ ]:
import os

print(os.path.exists("/content/app.py"))

True


In [ ]:
import threading
import time
from pyngrok import ngrok
from app import app

# Close any old ngrok tunnels
ngrok.kill()

# Start Flask in the background
def run_flask():
    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )

flask_thread = threading.Thread(
    target=run_flask,
    daemon=True
)
flask_thread.start()

# Allow Flask to start
time.sleep(3)

# Create a public website link
public_url = ngrok.connect(5000).public_url

print("Your Banknote Detection Website:")
print(public_url)

 * Serving Flask app 'app'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


Your Banknote Detection Website:
https://canned-antler-outburst.ngrok-free.dev


In [ ]:
print("Model Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Model Accuracy: 0.9745454545454545

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.95      0.98       153
           1       0.95      1.00      0.97       122

    accuracy                           0.97       275
   macro avg       0.97      0.98      0.97       275
weighted avg       0.98      0.97      0.97       275



In [ ]:
# Test the trained Linear SVM model

sample = [[3.6216, 8.6661, -2.8073, -0.44699]]

prediction = int(model.predict(sample)[0])

if prediction == 0:
    print("Prediction: Genuine Banknote")
else:
    print("Prediction: Counterfeit Banknote")

Prediction: Genuine Banknote


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [ ]:
import os

files = [
    "/content/app.py",
    "/content/banknote_model.pkl",
    "/content/templates/index.html"
]

for file in files:
    status = "FOUND" if os.path.exists(file) else "MISSING"
    print(file, ":", status)

/content/app.py : FOUND
/content/banknote_model.pkl : FOUND
/content/templates/index.html : FOUND
